# 🧘 Yoga Pose Similarity Search (Text + Image Search)
### Multimodal embeddings with CLIP + Milvus Vector Database

This notebook uses **CLIP** so you can search with:
- **Image** → similar images
- **Text** → relevant images
- **Image + Text** → weighted fusion query

All data + Milvus DB + checkpoints are stored on Google Drive.

**Dataset used:** `arrowe/yoga-poses-dataset-107` (downloaded as a zip to Drive).

---
## Cell 1 — Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# ⚙️ Change this if you want a different folder
DRIVE_ROOT = '/content/drive/MyDrive/YogaPoseSearch'

DRIVE_DATA       = os.path.join(DRIVE_ROOT, 'dataset')
DRIVE_DB         = os.path.join(DRIVE_ROOT, 'yoga_milvus.db')
DRIVE_CHECKPOINT = os.path.join(DRIVE_ROOT, 'checkpoint.txt')

# Dataset ZIP (downloaded from Kaggle endpoint you provided)
DATASET_ZIP = '/content/drive/MyDrive/computer-vision/yoga-poses-dataset-107.zip'

for d in [DRIVE_ROOT, DRIVE_DATA, os.path.dirname(DATASET_ZIP)]:
    os.makedirs(d, exist_ok=True)

print('Google Drive mounted ✓')
print('Drive root  :', DRIVE_ROOT)
print('Dataset dir :', DRIVE_DATA)
print('Milvus DB   :', DRIVE_DB)
print('Checkpoint  :', DRIVE_CHECKPOINT)
print('Dataset zip :', DATASET_ZIP)

---
## Cell 2 — Install Dependencies

In [ ]:
!pip install -q milvus-lite pymilvus open_clip_torch torch torchvision pillow flask pyngrok

---
## Cell 3 — Download + Unzip Dataset (Drive-backed)

Uses the download command you gave (Kaggle API endpoint via curl).

> Note: if Kaggle requires authentication/cookies in your session, this curl may download an HTML login page instead of the zip. In that case, use a Kaggle API token (`kaggle datasets download ...`) or manually download the zip to Drive.

In [ ]:
import glob
import zipfile

# 1) Download zip to Drive if missing
if not os.path.exists(DATASET_ZIP):
    print('Downloading dataset zip to Drive...')
    !curl -L https://www.kaggle.com/api/v1/datasets/download/arrowe/yoga-poses-dataset-107 -o "{DATASET_ZIP}"
    print('Download complete ✓')
else:
    print('Zip already exists on Drive — skipping download ✓')

# 2) Unzip to DRIVE_DATA if not already unzipped
train_dirs = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
if not train_dirs:
    print('Unzipping dataset into:', DRIVE_DATA)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(DRIVE_DATA)
    print('Unzip complete ✓')
else:
    print('Dataset appears already unzipped — skipping unzip ✓')

# 3) Find dataset train directory
train_dirs = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
DATASET_DIR = train_dirs[0] if train_dirs else DRIVE_DATA
print('Dataset root:', DATASET_DIR)
print('Classes found:', len([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR,d))]))

---
## Cell 4 — ⚙️ Configuration

In [ ]:
DB_PATH    = DRIVE_DB
COLLECTION = 'yoga_clip_embeddings'

# CLIP embedding dim for ViT-B-32 is 512
EMB_DIM = 512

BATCH_SIZE = 128
TOP_N = 5

print('Config set ✓')
print('DB_PATH   :', DB_PATH)
print('COLLECTION:', COLLECTION)
print('EMB_DIM   :', EMB_DIM)

---
## Cell 5 — Load CLIP (Image+Text) Model

In [ ]:
import torch
import open_clip
from PIL import Image
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

print('CLIP loaded ✓')

---
## Cell 6 — Embedding Functions (Image + Text)

In [ ]:
def l2_normalize(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x) + 1e-10)

def get_image_embedding(image_path: str) -> np.ndarray:
    img = Image.open(image_path).convert('RGB')
    img_t = clip_preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(img_t).squeeze().cpu().numpy()
    return l2_normalize(emb).astype(np.float32)

def get_text_embedding(text: str) -> np.ndarray:
    tokens = clip_tokenizer([text]).to(device)
    with torch.no_grad():
        emb = clip_model.encode_text(tokens).squeeze().cpu().numpy()
    return l2_normalize(emb).astype(np.float32)

print('Embedding functions ready ✓')

---
## Cell 7 — Build / Resume Milvus DB (Store Image Vectors)

In [ ]:
from pymilvus import MilvusClient
import time

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Load checkpoint
if os.path.exists(DRIVE_CHECKPOINT):
    with open(DRIVE_CHECKPOINT, 'r') as fh:
        already_done = set(line.strip() for line in fh if line.strip())
    print('Checkpoint loaded ✓', len(already_done))
else:
    already_done = set()
    print('No checkpoint found — starting fresh.')

client = MilvusClient(DB_PATH)

# Create collection if missing
if not client.has_collection(COLLECTION):
    client.create_collection(collection_name=COLLECTION, dimension=EMB_DIM, metric_type='COSINE', auto_id=True)
    print('Created collection', COLLECTION)

# Collect images
all_images = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_DIR)
    for f in files
    if os.path.splitext(f)[1].lower() in VALID_EXTS
]
pending = [p for p in all_images if p not in already_done]
print('Total images:', len(all_images), '| Pending:', len(pending))

data_buffer, done_buffer = [], []
inserted, skipped = 0, 0
start = time.time()
checkpoint_fh = open(DRIVE_CHECKPOINT, 'a')

try:
    for i, fpath in enumerate(pending):
        label = os.path.basename(os.path.dirname(fpath))
        try:
            emb = get_image_embedding(fpath)
        except Exception:
            skipped += 1
            continue

        data_buffer.append({'vector': emb, 'label': label, 'image_path': fpath})
        done_buffer.append(fpath)

        if len(data_buffer) >= BATCH_SIZE:
            client.insert(COLLECTION, data_buffer)
            checkpoint_fh.write('\n'.join(done_buffer) + '\n')
            checkpoint_fh.flush()
            inserted += len(data_buffer)
            data_buffer, done_buffer = [], []

        if (i + 1) % 500 == 0:
            print(f'[{i+1}/{len(pending)}] inserted={inserted} skipped={skipped} elapsed={time.time()-start:.0f}s')

    if data_buffer:
        client.insert(COLLECTION, data_buffer)
        checkpoint_fh.write('\n'.join(done_buffer) + '\n')
        checkpoint_fh.flush()
        inserted += len(data_buffer)
finally:
    checkpoint_fh.close()
    client.close()

print('✅ Build complete. Inserted:', inserted, 'Skipped:', skipped, 'Time:', round(time.time()-start,1),'s')

---
## Cell 8 — Search APIs (Image / Text / Image+Text)

In [ ]:
def _milvus_search(query_vec: np.ndarray, top_n: int = TOP_N):
    c = MilvusClient(DB_PATH)
    res = c.search(collection_name=COLLECTION, data=[query_vec.astype(np.float32)], limit=top_n,
                   output_fields=['label','image_path'], search_params={'metric_type':'COSINE'})
    c.close()
    return [{'score': round(float(h['distance']),4), 'label': h['entity']['label'], 'image_path': h['entity']['image_path']}
            for h in res[0]]

def search_by_image(image_path: str, top_n: int = TOP_N):
    return _milvus_search(get_image_embedding(image_path), top_n)

def search_by_text(text: str, top_n: int = TOP_N):
    return _milvus_search(get_text_embedding(text), top_n)

def search_by_image_and_text(image_path: str, text: str, alpha: float = 0.5, top_n: int = TOP_N):
    img_v = get_image_embedding(image_path)
    txt_v = get_text_embedding(text)
    q = alpha * img_v + (1.0 - alpha) * txt_v
    q = l2_normalize(q).astype(np.float32)
    return _milvus_search(q, top_n)

print('Search functions ready ✓')

---
## Cell 9 — Quick Demo: Text → Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

query = 'a person doing downward dog yoga pose'
hits = search_by_text(query, top_n=5)
print('Query:', query)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, hit in zip(axes, hits):
    ax.imshow(mpimg.imread(hit['image_path']))
    ax.set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()